In [5]:
import cv2
import os
import glob
import numpy as np
import matplotlib.pyplot as plt


video_path = r"D:\Downloads\IP_Project\Road_Damage.mp4"
base_dir   = r"D:\Downloads\IP_Project\Enhacement"

extract_dir = os.path.join(base_dir, "Extracted_frames")
final_dir   = os.path.join(base_dir, "Enhanced_frames")
compare_dir = os.path.join(base_dir, "Comparison_results")
log_dir     = os.path.join(base_dir, "Adaptive_logs")

for folder in [extract_dir, final_dir, compare_dir, log_dir]:
    os.makedirs(folder, exist_ok=True)


#-------- FRAME EXTRACTION-----------------

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("ERROR: Cannot open video.")
    exit()

frame_count = 0
saved_count = 0
frame_skip = 1

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1

    if frame_count % frame_skip != 0:
        continue

    frame = cv2.resize(frame, (640, 480))

    saved_count += 1
    cv2.imwrite(os.path.join(extract_dir, f"frame_{saved_count:03d}.jpg"), frame)

cap.release()

print("Extracted frames:", saved_count)


# ---------FRAME DEGRADATION ANALYSIS FUNCTIONS-----------

def analyse_frame(gray):

    mean_brightness = np.mean(gray)
    contrast = np.std(gray)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()

    # Noise estimation
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    noise_level = np.std(gray.astype("float") - blur.astype("float"))

    return mean_brightness, contrast, sharpness, noise_level


def decide_filters(mean_brightness, contrast, sharpness, noise_level):
   
    filters = []

    # Noise condition
    if noise_level > 8:
        filters.append("Median Filter")

    # Low contrast condition
    if contrast < 45:
        filters.append("CLAHE")

    # Too dark or too bright condition
    if mean_brightness < 70 or mean_brightness > 190:
        filters.append("Brightness Correction")

    # Blur condition
    if sharpness < 80:
        filters.append("Sharpening")

    return filters



# ------- ENHANCEMENT PIPELINE------------

frames = sorted(glob.glob(os.path.join(extract_dir, "*.jpg")))

log_path = os.path.join(log_dir, "adaptive_filter_log.txt")

with open(log_path, "w") as log_file:

    log_file.write("Frame, Brightness, Contrast, Sharpness, Noise, Applied Filters\n")

    for idx, path in enumerate(frames, start=1):

        original = cv2.imread(path)

        if original is None:
            continue

        gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

        mean_brightness, contrast, sharpness, noise_level = analyse_frame(gray)

        filters = decide_filters(
            mean_brightness,
            contrast,
            sharpness,
            noise_level
        )

        enhanced = gray.copy()

        # -------------------------------------------------
        # Apply only selected filters
        # -------------------------------------------------

        # 1. Median Filter (if noise is high)
        if "Median Filter" in filters:
            enhanced = cv2.medianBlur(enhanced, 5)

        # 2. Brightness Correction (if too dark/bright)
        if "Brightness Correction" in filters:

            if mean_brightness < 70:
                enhanced = cv2.convertScaleAbs(
                    enhanced,
                    alpha=1.1,
                    beta=15
                )

            elif mean_brightness > 190:
                enhanced = cv2.convertScaleAbs(
                    enhanced,
                    alpha=0.85,
                    beta=-20
                )

        # 3. CLAHE (if contrast is low)
        if "CLAHE" in filters:
            clahe = cv2.createCLAHE(
                clipLimit=1.2,
                tileGridSize=(8, 8)
            )
            enhanced = clahe.apply(enhanced)

        # 4. Sharpening (if frame is blur)
        if "Sharpening" in filters:
            blur = cv2.GaussianBlur(enhanced, (0, 0), 1.0)
            enhanced = cv2.addWeighted(
                enhanced, 1.2,
                blur, -0.2,
                0
            )

        # -------------------------------------------------
        # Save final enhanced frame
        # -------------------------------------------------
        final_path = os.path.join(final_dir, f"adaptive_enhanced_{idx:03d}.jpg")
        cv2.imwrite(final_path, enhanced)

        # -------------------------------------------------
        # Save comparison image
        # -------------------------------------------------
        plt.figure(figsize=(10, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(gray, cmap="gray")
        plt.title("Original Gray")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(enhanced, cmap="gray")
        plt.title("Adaptive Enhanced")
        plt.axis("off")

        plt.suptitle("Filters Used: " + ", ".join(filters) if filters else "Filters Used: None")

        plt.tight_layout()

        compare_path = os.path.join(compare_dir, f"adaptive_comparison_{idx:03d}.jpg")
        plt.savefig(compare_path, dpi=150)
        plt.close()

        # -------------------------------------------------
        # Write log
        # -------------------------------------------------
        log_file.write(
            f"frame_{idx:03d}, "
            f"{mean_brightness:.2f}, "
            f"{contrast:.2f}, "
            f"{sharpness:.2f}, "
            f"{noise_level:.2f}, "
            f"{filters}\n"
        )

print("Adaptive enhancement completed.")
print("Final frames saved in:", final_dir)
print("Filter log saved in:", log_path)

Extracted frames: 555
Adaptive enhancement completed.
Final frames saved in: D:\Downloads\IP_Project\Enhacement\Enhanced_frames
Filter log saved in: D:\Downloads\IP_Project\Enhacement\Adaptive_logs\adaptive_filter_log.txt


In [3]:
import cv2
import os
import glob

# =====================================================
# PATHS
# =====================================================
base_dir = r"D:\Downloads\IP_Project\Enhacement"

frame_dir = os.path.join(base_dir, "Enhanced_frames")
output_video = os.path.join(base_dir, "enhanced_video.mp4")

# =====================================================
# READ FRAMES
# =====================================================
frames = sorted(glob.glob(os.path.join(frame_dir, "*.jpg")))

print("Total frames found:", len(frames))

if len(frames) == 0:
    print("No enhanced frames found.")
    exit()

# =====================================================
# GET FRAME SIZE
# =====================================================
first_frame = cv2.imread(frames[0])
height, width, layers = first_frame.shape

# =====================================================
# VIDEO SETTINGS
# =====================================================
fps = 30   # change to 20 or 25 if video is too fast

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

# =====================================================
# WRITE FRAMES TO VIDEO
# =====================================================
for frame_path in frames:
    frame = cv2.imread(frame_path)

    if frame is None:
        print("Skipped:", frame_path)
        continue

    frame = cv2.resize(frame, (width, height))
    video.write(frame)

video.release()

print("Enhanced video created successfully.")
print("Saved video path:", output_video)

Total frames found: 555
Enhanced video created successfully.
Saved video path: D:\Downloads\IP_Project\Enhacement\enhanced_video.mp4
